In [1]:
from pathlib import Path
import json
import os
import re

import pandas as pd
import requests
from dotenv import load_dotenv

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "_quarto.yml").exists()
    ),
    None,
)

if ROOT is None:
    raise RuntimeError("Open this notebook from inside your project folder.")

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "outputs"

for folder in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", ROOT)

Project folder: /home/ubuntu/ad688-career-evaluation-group9


In [2]:
load_dotenv(ROOT / ".env")

API_KEY = os.getenv("EMPLOYABILITY_API_KEY", "").strip()
BASE_URL = os.getenv(
    "EMPLOYABILITY_API_BASE_URL",
    "https://met-employability-services.azurewebsites.net",
).strip().rstrip("/")

PATH_PREFIX = os.getenv(
    "EMPLOYABILITY_API_PATH_PREFIX",
    "api/v1/student",
).strip().strip("/")

if not API_KEY:
    raise RuntimeError("Add EMPLOYABILITY_API_KEY to your .env file.")

API_URL = f"{BASE_URL}/{PATH_PREFIX}"

HEADERS = {
    "X-API-Key": API_KEY,
    "Accept": "application/json",
}

In [8]:
response = requests.get(
    f"{API_URL}/jobs/",
    headers=HEADERS,
    params={
        "naics": "5132",
        "q": "data scientist",
        "limit": 10,
        "offset": 0,
    },
    timeout=180,
)

response.raise_for_status()

test_payload = response.json()
test_df = pd.json_normalize(test_payload["results"])

print("Rows returned:", len(test_df))
display(test_df[["job_id", "title", "company_name"]])

Rows returned: 10


,job_id,title,company_name
0,783636,Data scientist,Katholieke Universiteit Te Leuven AV
1,1679378,Data Scientist / Lab Specialist m/w/d - Roche Penzberg (Data Scientist),Franz & Wach Personalservice PEN
2,814504,Senior Software and AI Engineer,Coreweave
3,824735,Marketing Analytics Manager,Redwood Software
4,824734,Marketing Analytics Analyst,Redwood Software
5,830950,"Job-30477 Software Architect / AI Engineer — SDLC Agêntico, Brazil",Ci And T
6,695086,AI Engineer (m/w/d) - LLM-Systeme & Softwareintegration,Macs Software GmbH
7,760836,Expert Fragrance Analytics (d/m/w) (Analytiker/in - Chemie),Henkel AG & Co. KGaA
8,1398143,Expert Fragrance Analytics (d/m/w),Henkel AG & Co. KGaA
9,1584590,Anwendungsberater - SAP / .NET / SQL-Server (m/w/d) (Anwendungsberater/in),Workwise GmbH


In [10]:
# Keep Data Scientist and closely related AI/ML roles

role_terms = [
    r"\bdata scientist\b",
    r"\bsenior data scientist\b",
    r"\bjunior data scientist\b",
    r"\blead data scientist\b",
    r"\bprincipal data scientist\b",
    r"\bmachine learning scientist\b",
    r"\bapplied scientist\b",
    r"\bresearch scientist\b",
    r"\bmachine learning engineer\b",
    r"\bml engineer\b",
    r"\bai engineer\b",
]

role_pattern = "|".join(role_terms)

test_df["title_clean"] = (
    test_df["title"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

test_df["role_match"] = test_df["title_clean"].str.contains(
    role_pattern,
    regex=True,
    na=False,
)

clean_df = test_df[test_df["role_match"]].copy()

print("Data Scientist and related-role matches:", len(clean_df))

display(
    clean_df[
        [
            "job_id",
            "title",
            "company_name",
            "naics_code",
            "naics_name",
        ]
    ]
)

Data Scientist and related-role matches: 5


,job_id,title,company_name,naics_code,naics_name
0,783636,Data scientist,Katholieke Universiteit Te Leuven AV,513200,Software Publishers
1,1679378,Data Scientist / Lab Specialist m/w/d - Roche Penzberg (Data Scientist),Franz & Wach Personalservice PEN,513200,Software Publishers
2,814504,Senior Software and AI Engineer,Coreweave,513200,Software Publishers
5,830950,"Job-30477 Software Architect / AI Engineer — SDLC Agêntico, Brazil",Ci And T,513200,Software Publishers
6,695086,AI Engineer (m/w/d) - LLM-Systeme & Softwareintegration,Macs Software GmbH,513200,Software Publishers


In [11]:
staffing_terms = [
    "staffing",
    "recruit",
    "recruiting",
    "talent",
    "employment",
    "personaldienst",
    "personalservice",
    "constaff",
    "akkodis",
    "orizon",
    "cure-x",
]

staffing_pattern = "|".join(
    re.escape(term)
    for term in staffing_terms
)

clean_df["company_clean"] = (
    clean_df["company_name"]
    .fillna("")
    .str.lower()
    .str.strip()
)

clean_df["company_name_staffing_flag"] = (
    clean_df["company_clean"]
    .str.contains(staffing_pattern, regex=True, na=False)
)

if "is_staffing_agency" in clean_df.columns:
    clean_df["api_staffing_flag"] = (
        clean_df["is_staffing_agency"]
        .fillna(False)
        .astype(bool)
    )
else:
    clean_df["api_staffing_flag"] = False

clean_df["staffing_flag"] = (
    clean_df["company_name_staffing_flag"]
    | clean_df["api_staffing_flag"]
)

clean_df = clean_df[~clean_df["staffing_flag"]].copy()

print("Rows after removing staffing agencies:", len(clean_df))

display(clean_df[["job_id", "title", "company_name"]])

Rows after removing staffing agencies: 4


,job_id,title,company_name
0,783636,Data scientist,Katholieke Universiteit Te Leuven AV
2,814504,Senior Software and AI Engineer,Coreweave
5,830950,"Job-30477 Software Architect / AI Engineer — SDLC Agêntico, Brazil",Ci And T
6,695086,AI Engineer (m/w/d) - LLM-Systeme & Softwareintegration,Macs Software GmbH


In [12]:
# Remove duplicates and clean date, salary, remote, and location fields

clean_df = clean_df.drop_duplicates(subset="job_id").copy()

for column in ["posted_at", "expires_at"]:
    if column in clean_df.columns:
        clean_df[column] = pd.to_datetime(
            clean_df[column],
            errors="coerce",
            utc=True,
        )

for column in [
    "annual_salary_min",
    "annual_salary_max",
    "hourly_salary_min",
    "hourly_salary_max",
]:
    if column in clean_df.columns:
        clean_df[column] = pd.to_numeric(
            clean_df[column],
            errors="coerce",
        )

if {
    "annual_salary_min",
    "annual_salary_max",
}.issubset(clean_df.columns):
    clean_df["annual_salary_midpoint"] = (
        clean_df["annual_salary_min"]
        + clean_df["annual_salary_max"]
    ) / 2

if "remote_status" in clean_df.columns:
    clean_df["remote_status"] = (
        clean_df["remote_status"]
        .fillna("unknown")
        .str.lower()
        .str.strip()
    )

for column in ["city", "state", "location_text"]:
    if column in clean_df.columns:
        clean_df[column] = (
            clean_df[column]
            .fillna("Unknown")
            .astype(str)
            .str.strip()
        )

In [13]:
# Convert skill lists into readable text

def format_skills(value):
    if not isinstance(value, list):
        return ""

    names = []

    for item in value:
        if isinstance(item, dict) and item.get("name"):
            names.append(str(item["name"]))
        elif isinstance(item, str):
            names.append(item)

    return ", ".join(names)

if "skills" in clean_df.columns:
    clean_df["skills_text"] = clean_df["skills"].apply(format_skills)
else:
    clean_df["skills_text"] = ""

In [14]:
# Save the cleaned dataset

columns_to_remove = [
    "company_clean",
    "company_name_staffing_flag",
    "api_staffing_flag",
]

clean_df = clean_df.drop(
    columns=[
        column
        for column in columns_to_remove
        if column in clean_df.columns
    ]
)

output_file = PROCESSED_DIR / "data_scientist_software_publishers.csv"

clean_df.to_csv(output_file, index=False)

print("Final rows:", len(clean_df))
print("Saved cleaned dataset to:", output_file)

display(
    clean_df[
        [
            column
            for column in [
                "job_id",
                "title",
                "company_name",
                "posted_at",
                "remote_status",
                "annual_salary_min",
                "annual_salary_max",
                "skills_text",
            ]
            if column in clean_df.columns
        ]
    ]
)

Final rows: 4
Saved cleaned dataset to: /home/ubuntu/ad688-career-evaluation-group9/data/processed/data_scientist_software_publishers.csv


,job_id,title,company_name,posted_at,remote_status,annual_salary_min,annual_salary_max,skills_text
0,783636,Data scientist,Katholieke Universiteit Te Leuven AV,2026-07-28 11:15:38.614000+00:00,unknown,NaN,NaN,Agentic AI
2,814504,Senior Software and AI Engineer,Coreweave,NaT,unknown,182000.0,242000.0,
5,830950,"Job-30477 Software Architect / AI Engineer — SDLC Agêntico, Brazil",Ci And T,NaT,unknown,0.0,0.0,
6,695086,AI Engineer (m/w/d) - LLM-Systeme & Softwareintegration,Macs Software GmbH,NaT,unknown,NaN,NaN,
